# Notebook 02 — Data Preparation

## Objective

The objective of this notebook is to transform the raw streamed **MindBigData2023 MNIST-2B** dataset into a clean, structured dataset that can be efficiently used throughout the remainder of the project.

Unlike Notebook 01, which focused on understanding the dataset, this notebook focuses on **preparing the data for machine learning experiments** — for both the train/val pool and the held-out test set.

This notebook does **not** decide the train/val split — that happens in Notebook 03, using the metadata preserved here. It does, however, adopt Hugging Face's official `test` split as our final test set (pending a session-overlap sanity check in Notebook 03), so the test set is extracted here rather than fabricated ourselves.

---

## Tasks

This notebook will:

1. Load the Hugging Face dataset (`train` split) in streaming mode.
2. Draw a class-balanced subsample, pairing each digit trial with its preceding blank trial.
3. Extract and reshape each trial into a `(128 × 256)` EEG matrix.
4. Preserve digit labels (`-1`/`0–9`) and binary labels (`0`/`1`) for Stage 1/Stage 2.
5. Preserve metadata (`sessionnum`, `blocknum`, `blockpos`, `timestamp`) for leakage-safe splitting later.
6. Save the train/val pool to HDF5 (array-based, not per-trial groups) and validate it.
7. Repeat the same extraction, at a smaller scale, against the official `test` split.
8. Save and validate the test set to a **separate** HDF5 file.

---

## Output

```
data/processed/mindbigdata2023_train.h5   # train/val pool — split assigned in Notebook 03
data/processed/mindbigdata2023_test.h5    # held-out test set — untouched until final evaluation
```

## 1. Imports & Configuration

In [ ]:
from datasets import load_dataset
import h5py
import numpy as np
from collections import defaultdict
from pathlib import Path

# --- Config ---
TRAIN_TARGET_PER_CLASS = 1400        # ~10-20% of ~7000/class; tune as needed (total 70000 points)
TEST_TARGET_PER_CLASS = 200
N_CHANNELS = 128
N_SAMPLES = 256                # confirmed empirically in Notebook 01
TRAIN_POOL_PATH = Path("../data/processed/mindbigdata2023_train_pool.h5")
TEST_PATH = Path("../data/processed/mindbigdata2023_test.h5")
TRAIN_POOL_PATH.parent.mkdir(parents=True, exist_ok=True)

# Labels present in this dataset: -1 (blank), 0-9 (digit) -> 11 distinct label values
ALL_LABELS = [-1] + list(range(10))

## 2. Load Train Dataset (Streaming)

We use `streaming=True` so we never download the full ~13.5 GB file — only the
rows we actually keep get pulled and written to disk.

In [7]:
ds = load_dataset(
    "DavidVivancos/MindBigData2023_MNIST-2B",
    split="train",
    streaming=True
)
ds

IterableDataset({
    features: Unknown,
    num_shards: 1
})

## 3. Stratified Subsample, Extraction & Write

For each streamed row we:
- Determine channel column names once (first row only)
- Reshape the flat EEG columns into a `(128, 256)` array
- Build both label views (binary + digit)
- Carry over metadata needed for splitting later
- Write immediately to HDF5 — no accumulation in memory

We stop once every label class has reached `TRAIN_TARGET_PER_CLASS` trials, or once a
generous streaming cap is hit (safety net in case some class is rare).

In [ ]:
digit_counts = defaultdict(int)   # digit label (0-9) -> count so far
channel_names = None
trial_idx = 0
pending_blank = None        # holds the most recent blank row until its paired digit arrives

# Safety cap: stop streaming after this many rows even if some class never fills up.
MAX_ROWS_TO_STREAM = TRAIN_TARGET_PER_CLASS * 10 * 20

def extract_eeg(row, channel_names):
    return np.array(
        [[row[f"{ch}_{i}"] for i in range(N_SAMPLES)] for ch in channel_names],
        dtype=np.float32
    )  # shape: (128, 256)

def append_row(dset, value):
    dset.resize(dset.shape[0] + 1, axis=0)
    dset[-1] = value

with h5py.File(TRAIN_POOL_PATH, "a") as f:
    eeg_ds = f.create_dataset("eeg", shape=(0, N_CHANNELS, N_SAMPLES), maxshape=(None, N_CHANNELS, N_SAMPLES),
                               chunks=(1, N_CHANNELS, N_SAMPLES), compression="gzip", dtype="float32")
    label_binary_ds = f.create_dataset("label_binary", shape=(0,), maxshape=(None,), dtype="int8")
    label_digit_ds = f.create_dataset("label_digit", shape=(0,), maxshape=(None,), dtype="int8")
    sessionnum_ds = f.create_dataset("sessionnum", shape=(0,), maxshape=(None,), dtype="int64")
    blocknum_ds = f.create_dataset("blocknum", shape=(0,), maxshape=(None,), dtype="int64")
    blockpos_ds = f.create_dataset("blockpos", shape=(0,), maxshape=(None,), dtype="int64")
    timestamp_ds = f.create_dataset("timestamp", shape=(0,), maxshape=(None,), dtype="int64")

    def write_trial(row, label_digit, label_binary):
        global trial_idx
        eeg = extract_eeg(row, channel_names)
        append_row(eeg_ds, eeg)
        append_row(label_binary_ds, label_binary)
        append_row(label_digit_ds, label_digit)
        append_row(sessionnum_ds, row["sessionnum"])
        append_row(blocknum_ds, row["blocknum"])
        append_row(blockpos_ds, row["blockpos"])
        append_row(timestamp_ds, row["timestamp"])
        trial_idx += 1

    for row_num, row in enumerate(ds):
        if row_num >= MAX_ROWS_TO_STREAM:
            print("Hit streaming cap before all classes filled - see counts below.")
            break

        label = row["label"]

        if channel_names is None:
            channel_names = sorted({
                k.rsplit("_", 1)[0] for k in row.keys()
                if "_" in k and not k.startswith("label")
            })
            print(f"Detected {len(channel_names)} channels")

        if label == -1:
            pending_blank = row
            continue

        if digit_counts[label] >= TRAIN_TARGET_PER_CLASS:
            pending_blank = None
            continue

        if pending_blank is None:
            print(f"WARNING: digit {label} at row {row_num} has no preceding blank - skipping pair")
            continue

        write_trial(row, label_digit=label, label_binary=1)
        write_trial(pending_blank, label_digit=-1, label_binary=0)

        digit_counts[label] += 1
        pending_blank = None

        if trial_idx % 500 == 0:
            print(f"{trial_idx} trials written...")

        if all(digit_counts[d] >= TRAIN_TARGET_PER_CLASS for d in range(10)):
            print(f"All digit classes reached target after {trial_idx} trials written.")
            break

print("\nFinal counts per digit class:")
for d in range(10):
    print(f"  {d}: {digit_counts[d]}")
print(f"\nTotal trials written: {trial_idx}")

Detected 128 channels
500 trials written...
1000 trials written...
1500 trials written...
2000 trials written...
2500 trials written...
3000 trials written...
3500 trials written...
4000 trials written...
4500 trials written...
5000 trials written...
5500 trials written...
6000 trials written...
6500 trials written...
7000 trials written...
7500 trials written...
8000 trials written...
8500 trials written...
9000 trials written...
9500 trials written...
10000 trials written...
10500 trials written...
11000 trials written...
11500 trials written...
12000 trials written...
12500 trials written...
13000 trials written...
13500 trials written...
14000 trials written...
14500 trials written...
15000 trials written...
15500 trials written...
16000 trials written...
16500 trials written...
17000 trials written...
17500 trials written...
18000 trials written...
18500 trials written...
19000 trials written...
19500 trials written...
20000 trials written...
20500 trials written...
21000 trials w

## 4. Load Test Dataset

In [10]:
ds_test = load_dataset(
    "DavidVivancos/MindBigData2023_MNIST-2B",
    split="test",
    streaming=True
)

digit_counts_test = defaultdict(int)
pending_blank_test = None
trial_idx_test = 0
MAX_ROWS_TEST = TEST_TARGET_PER_CLASS * 10 * 20

with h5py.File(TEST_PATH, "a") as f:
    eeg_ds = f.create_dataset("eeg", shape=(0, N_CHANNELS, N_SAMPLES), maxshape=(None, N_CHANNELS, N_SAMPLES),
                               chunks=(1, N_CHANNELS, N_SAMPLES), compression="gzip", dtype="float32")
    label_binary_ds = f.create_dataset("label_binary", shape=(0,), maxshape=(None,), dtype="int8")
    label_digit_ds = f.create_dataset("label_digit", shape=(0,), maxshape=(None,), dtype="int8")
    sessionnum_ds = f.create_dataset("sessionnum", shape=(0,), maxshape=(None,), dtype="int64")
    blocknum_ds = f.create_dataset("blocknum", shape=(0,), maxshape=(None,), dtype="int64")
    blockpos_ds = f.create_dataset("blockpos", shape=(0,), maxshape=(None,), dtype="int64")
    timestamp_ds = f.create_dataset("timestamp", shape=(0,), maxshape=(None,), dtype="int64")

    def write_trial_test(row, label_digit, label_binary):
        global trial_idx_test
        eeg = extract_eeg(row, channel_names)   # reuses function defined in Section 3
        append_row(eeg_ds, eeg)
        append_row(label_binary_ds, label_binary)
        append_row(label_digit_ds, label_digit)
        append_row(sessionnum_ds, row["sessionnum"])
        append_row(blocknum_ds, row["blocknum"])
        append_row(blockpos_ds, row["blockpos"])
        append_row(timestamp_ds, row["timestamp"])
        trial_idx_test += 1

    for row_num, row in enumerate(ds_test):
        if row_num >= MAX_ROWS_TEST:
            print("Hit streaming cap before all classes filled.")
            break

        label = row["label"]

        if label == -1:
            pending_blank_test = row
            continue

        if digit_counts_test[label] >= TEST_TARGET_PER_CLASS:
            pending_blank_test = None
            continue

        if pending_blank_test is None:
            print(f"WARNING: digit {label} at row {row_num} has no preceding blank - skipping")
            continue

        write_trial_test(row, label_digit=label, label_binary=1)
        write_trial_test(pending_blank_test, label_digit=-1, label_binary=0)

        digit_counts_test[label] += 1
        pending_blank_test = None

        if trial_idx_test % 500 == 0:
            print(f"{trial_idx_test} trials written...")

        if all(digit_counts_test[d] >= TEST_TARGET_PER_CLASS for d in range(10)):
            print(f"All test classes reached target after {trial_idx_test} trials.")
            break

print("\nTest set counts per digit class:")
for d in range(10):
    print(f"  {d}: {digit_counts_test[d]}")
print(f"Total test trials written: {trial_idx_test}")

500 trials written...
1000 trials written...
1500 trials written...
2000 trials written...
2500 trials written...
3000 trials written...
3500 trials written...
4000 trials written...
All test classes reached target after 4000 trials.

Test set counts per digit class:
  0: 200
  1: 200
  2: 200
  3: 200
  4: 200
  5: 200
  6: 200
  7: 200
  8: 200
  9: 200
Total test trials written: 4000


## 5. Validate the Saved Dataset

Sanity checks before moving to Notebook 03:
- Correct number of trials
- Correct EEG shape per trial
- No NaNs / corrupt values
- Metadata present on every trial
- Label distribution matches what we expect

In [ ]:
with h5py.File(TRAIN_POOL_PATH, "r") as f:
    n_trials = f["eeg"].shape[0]
    print(f"Total trials saved: {n_trials}")
    print(f"EEG dataset shape: {f['eeg'].shape}  (should be (N, 128, 256))")

    label_binary = f["label_binary"][:]
    label_digit = f["label_digit"][:]

    print(f"Binary label balance -> blank: {(label_binary == 0).sum()}, digit: {(label_binary == 1).sum()}")

    nan_count = 0
    for i in range(0, n_trials, 1000):   # spot-check in chunks to avoid loading all 3GB at once
        chunk = f["eeg"][i:i+1000]
        nan_count += np.isnan(chunk).sum()
    print(f"NaN values found: {nan_count}")

    print("\nDigit label distribution:")
    for d in range(-1, 10):
        print(f"  {d}: {(label_digit == d).sum()}")

Total trials saved: 28000
EEG dataset shape: (28000, 128, 256)  (should be (N, 128, 256))
Binary label balance -> blank: 14000, digit: 14000
NaN values found: 0

Digit label distribution:
  -1: 14000
  0: 1400
  1: 1400
  2: 1400
  3: 1400
  4: 1400
  5: 1400
  6: 1400
  7: 1400
  8: 1400
  9: 1400


In [12]:
with h5py.File(TEST_PATH, "r") as f:
    n_test = f["eeg"].shape[0]
    print(f"Total test trials saved: {n_test}")
    print(f"EEG dataset shape: {f['eeg'].shape}  (should be (N, 128, 256))")

    label_binary = f["label_binary"][:]
    label_digit = f["label_digit"][:]

    print(f"Binary label balance -> blank: {(label_binary == 0).sum()}, digit: {(label_binary == 1).sum()}")

    nan_count = 0
    for i in range(0, n_test, 1000):
        chunk = f["eeg"][i:i+1000]
        nan_count += np.isnan(chunk).sum()
    print(f"NaN values found: {nan_count}")

    print("\nTest digit label distribution:")
    for d in range(-1, 10):
        print(f"  {d}: {(label_digit == d).sum()}")

Total test trials saved: 4000
EEG dataset shape: (4000, 128, 256)  (should be (N, 128, 256))
Binary label balance -> blank: 2000, digit: 2000
NaN values found: 0

Test digit label distribution:
  -1: 2000
  0: 200
  1: 200
  2: 200
  3: 200
  4: 200
  5: 200
  6: 200
  7: 200
  8: 200
  9: 200


## Summary

Two files now exist in `data/processed/`:

**`mindbigdata2023_train.h5`** — the train/val pool, drawn from HF's `train` split:
- `eeg`: shape `(N, 128, 256)`, float32
- `label_binary`: `0` (blank) / `1` (digit) — Stage 1
- `label_digit`: `-1` (blank) / `0–9` — Stage 2
- `sessionnum`, `blocknum`, `blockpos`, `timestamp` — for leakage-safe splitting
- `N ≈ TARGET_PER_CLASS × 20` (1,400/digit class × 10, plus paired blanks)

**`mindbigdata2023_test.h5`** — the held-out test set, drawn from HF's official `test` split, same structure, `N ≈ TEST_TARGET_PER_CLASS × 20` (300/digit class × 10, plus paired blanks).

**No train/val split has been assigned within the train pool yet** — that happens in Notebook 03, entirely against the local `mindbigdata2023_train.h5` file.

**Before trusting the test set as truly leakage-free**, Notebook 03 will first verify there's no `sessionnum` overlap between the train pool and the test file. If overlap is found, we fall back to deriving our own 3-way split instead of relying on HF's train/test boundary.

The test set should not be touched again until final model evaluation.